In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\koush\Desktop\GenAi project folder\all_features.csv")

df = df.drop(columns=["file","client_ip","app","category","mode","run_id",])

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report


In [3]:
y = (df["label"] == "genai").astype(int).values

# keep numeric features only
X = df.drop(columns=["label"]).select_dtypes(include="number")
feature_names = X.columns.tolist()


In [4]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [5]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"   
)


In [6]:
rf_scores = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")
print("RF CV scores:", rf_scores)
print("RF mean:", rf_scores.mean(), "std:", rf_scores.std())

RF CV scores: [0.81818182 0.90909091 0.90909091 0.90909091 0.81818182 0.90909091
 0.8        0.9        0.9        0.9       ]
RF mean: 0.8772727272727273 std: 0.04307402814720773


In [7]:
y_pred_rf = cross_val_predict(rf, X, y, cv=cv)

print("RF Confusion Matrix:\n", confusion_matrix(y, y_pred_rf))
print("\nRF Classification Report:\n")
print(classification_report(y, y_pred_rf, target_names=["non_genai", "genai"]))

RF Confusion Matrix:
 [[47  4]
 [ 9 46]]

RF Classification Report:

              precision    recall  f1-score   support

   non_genai       0.84      0.92      0.88        51
       genai       0.92      0.84      0.88        55

    accuracy                           0.88       106
   macro avg       0.88      0.88      0.88       106
weighted avg       0.88      0.88      0.88       106



In [8]:
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("\nTop Feature 15 Importances:\n", importances.head(15))


Top Feature 15 Importances:
 unique_src_ips      0.182950
unique_dst_ips      0.124684
dl_bytes            0.066517
dl_pkt_size_mean    0.046207
ul_p95_p50          0.040661
ul_pkt_size_p50     0.033824
iat_mean            0.031380
dl_pkt_size_p50     0.029048
total_packets       0.028491
bytes_per_second    0.027073
ul_pkt_size_std     0.026469
dl_packets          0.026403
ul_bytes            0.025692
pkt_size_mean       0.023205
dl_iat_mean         0.022692
dtype: float64
